# Text EDA: live coding

**Question:** Which highlighted conference messages express similar ideas, and how do their years and recorded speaker roles relate to those patterns?

We will build one step at a time: a phrase, a kicker, all kickers, a feature matrix, PHATE coordinates, and progressively more informative plots.

Before class, run `uv sync`, start the Ollama application, and run `ollama pull nomic-embed-text:v1.5`. Open this project folder in Positron and choose its `.venv` notebook kernel. All file paths below are relative to that folder.

The embedding calls use your local Nomic model. Model results will appear when you run the notebook.

**Teaching route:** Sections 1–18 build the main demonstration. Section 19 is an optional refit extension. Finish with the discussion, then use the save-vectors and [Plotly publishing](#publish-plotly) sections as needed.

Run cells from top to bottom. In the live scaffold, complete each TODO before running the cells that use its result. To prepare student practice, run the save-vectors cell once and share `data/talk_embeddings.npz` with the project.


## 1. Embed one simple phrase

Nomic uses task prefixes. `clustering: ` asks for representations useful for grouping texts by common content. It does not run a clustering algorithm. We will use the same prefix throughout.

In [ ]:
import ollama

MODEL = "nomic-embed-text:v1.5"
phrase = "Helping a neighbor through a difficult time."

# TODO: Call ollama.embed with model=MODEL,
# input="clustering: " + phrase, and truncate=False.
# Store the result as phrase_response.

## 2. Inspect what came back

The response contains a list of embeddings. One input still gives an outer list, so `[0]` selects its vector. Inspect a few values before doing anything else.

In [ ]:
print(type(phrase_response))
print("Number of returned embeddings:", len(phrase_response["embeddings"]))
phrase_vector = phrase_response["embeddings"][0]
print("Vector length:", len(phrase_vector))
phrase_vector[:8]

## 3. Predict, then repeat the same call

Which two phrases should be most similar? What might happen with the negated sentence? Predict before running. The model supplies a learned representation; it may not handle every comparison as you expect.

In [ ]:
phrases = [
    "Helping a neighbor through a difficult time.",
    "Offering support to someone facing hardship.",
    "The train arrives at the station at noon.",
    "Refusing to help a neighbor through a difficult time.",
]

phrase_vectors = []
# TODO: Loop over phrases. Repeat the same embedding call for each text.
# Append each returned vector to phrase_vectors.

## 4. Connect cosine similarity to the small example

For u=(1,0) and v=(1,1), the dot product is 1 and the product of lengths is sqrt(2). The result should be about 0.707. What would perpendicular or opposite vectors give?

In [ ]:
import numpy as np

u = np.array([1, 0])
v = np.array([1, 1])

numerator = np.dot(u, v)
denominator = np.linalg.norm(u) * np.linalg.norm(v)
numerator / denominator

## 5. Compare the phrase embeddings

Use the same cosine calculation through a library function. It works across all the coordinates of each model vector. Scores are not probabilities that statements have identical meaning.

In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

phrase_matrix = np.array(phrase_vectors)
print("Phrases by embedding features:", phrase_matrix.shape)

similarities = cosine_similarity(phrase_matrix)
pd.DataFrame(similarities, index=range(1, 5), columns=range(1, 5)).round(3)

**Pause:** Which result agreed with your prediction? Which result needs a closer reading of the phrases?

## 6. Look at one conference talk

Each row represents a talk through its **kicker**, the highlighted message before its body. We use a small selected sample of 52 kickers from six conferences in 2020, 2023, and 2025. It is not a census or random sample. A kicker does not describe every theme in a full talk. Each row includes its official source URL.


In [ ]:
df = pd.read_csv("data/talks.csv")
df[["title", "speaker", "year", "kicker"]].head()

In [ ]:
talk = df.iloc[0]
print(talk["title"])
print(talk["kicker"])

## 7. Embed that one kicker

This is the same operation as embedding a phrase. Titles, names, dates, and recorded roles stay out of the model input.

In [ ]:
# TODO: Embed talk["kicker"] using the same model and prefix.
# Store its returned vector as talk_vector.
# Inspect its length and first few values.

## 8. Repeat for all of our kickers

A plain loop makes the repeated operation visible. We keep the DataFrame order so every vector stays attached to the correct talk.

In [ ]:
talk_vectors = []

# TODO: Loop over df["kicker"], embed each text,
# and append its vector to talk_vectors.

print("Kickers embedded:", len(talk_vectors))

## 9. Inspect the DataFrame with its new column

What does one cell of the embedding column contain? Is this table an effective way to examine relationships among the talks?

In [ ]:
df["embedding"] = talk_vectors
df[["title", "kicker", "embedding"]].head()

**Next question:** How could we visualize observations that each have hundreds of numerical features?

## 10. Make the feature matrix

Stack the vectors into rows. The DataFrame has one column of lists; the feature matrix has one column for each embedding coordinate. Keep the model vectors on their original scale; z-scoring individual coordinates would change their geometry. Unlike named variables such as age and fare, these coordinates are learned together by the model.


In [ ]:
# TODO: Stack the embedding vectors into a matrix named X.
# Inspect X.shape and compare it with df.shape.

## 11. Apply PHATE using its defaults

The familiar `fit_transform` pattern now receives text-derived features. We fix only the random seed and leave the other settings at their defaults.

There are two different representations here: each **model embedding** has hundreds of features; each **PHATE point** has two plotting coordinates. PHATE axes do not have topic names, and a visible gap does not by itself establish separate groups. Use the map to choose texts to inspect.


In [ ]:
import phate

# TODO: Create phate.PHATE(random_state=386), then fit_transform(X).
# Store the two-dimensional result as coordinates.

In [ ]:
df["PHATE1"] = coordinates[:, 0]
df["PHATE2"] = coordinates[:, 1]
df[["title", "PHATE1", "PHATE2"]].head()

## 12. Start with the simplest scatterplot

One point is one kicker. What can this view tell us? What is still missing for our question?

In [ ]:
import matplotlib.pyplot as plt

# TODO: Make a basic scatterplot of PHATE1 and PHATE2.
# Label the axes and show the plot.

## 13. Add one layer: color by year

Do messages from different years overlap in this view? Year is metadata and was not part of the embedding input. Describe what the plot suggests, then consider what actual kickers you would need to read.

In [ ]:
plt.scatter(df["PHATE1"], df["PHATE2"], c=df["year"])
plt.colorbar(label="Conference year")
plt.xlabel("PHATE 1")
plt.ylabel("PHATE 2")
plt.show()

## 14. Make the same view interactive

Keep the same coordinates and year colors. Plotly adds pan, zoom, and hover. At first, the hover contains mostly coordinates; we will improve it next.

In [ ]:
import plotly.express as px

fig = px.scatter(df, x="PHATE1", y="PHATE2", color="year")
fig.show()

## 15. Add useful hover information

Which actual messages created a pattern? Read a few kickers, including one that complicates your explanation. Use the source URL in df if you need to inspect the full talk.
Nearby points suggest a comparison to investigate. Read their messages before naming a theme, and look for a nearby kicker that does not fit. Hovering helps connect the compressed view back to the original data.


In [ ]:
# TODO: Repeat the previous Plotly call.
# Add hover_name="title" and useful hover_data, including the kicker.

## 16. Ask another question: recorded speaker roles

Change only the color variable. Do nearby messages come from speakers with different recorded roles? Read their kickers to support the interpretation. The roles describe the source talk pages, not current positions.

In [ ]:
fig = px.scatter(
    df, x="PHATE1", y="PHATE2", color="role_group",
    hover_name="title",
    hover_data=["speaker", "speaker_role", "kicker"],
)
fig.show()

## 17. Add another visual channel: marker shape

Does conference month add useful information to the role comparison? Add symbol without changing the coordinates or role colors. In this table, 4 means April and 10 means October. If the extra encoding does not help the question, leave it out.

In [ ]:
full_map = px.scatter(
    df, x="PHATE1", y="PHATE2", color="role_group", symbol="month",
    hover_name="title",
    hover_data=["speaker", "speaker_role", "year", "kicker"],
)
full_map.show()

## 18. Filter the displayed talks

Suppose we want to inspect 2023 and later. Selecting rows keeps their existing PHATE coordinates. No model or dimensionality reduction method is refitted.
To inspect one speaker instead, change the condition to, for example, `keep = df["speaker"] == "Russell M. Nelson"`, then rerun the filtering and plotting cells. This displays only five talks in this sample, so use the larger year subset for the optional refit demonstration.


In [ ]:
keep = df["year"] >= 2023
recent = df.loc[keep].copy()
print("Displayed talks:", len(recent), "of", len(df))
recent[["title", "year", "PHATE1", "PHATE2"]].head()

Keep the original axis ranges and year scale so the filtered view uses the same visual reference frame. Otherwise automatic rescaling could make the same coordinates or years look different.


In [ ]:
fig = px.scatter(
    recent, x="PHATE1", y="PHATE2", color="year",
    hover_name="title", hover_data=["speaker", "kicker"],
    range_x=[df["PHATE1"].min(), df["PHATE1"].max()],
    range_y=[df["PHATE2"].min(), df["PHATE2"].max()],
    range_color=[df["year"].min(), df["year"].max()],
)
fig.show()

**Investigate:** Choose a question about the year or role comparisons. Read at least three kickers, identify a complication, and state one next check. The selected sample cannot establish historical trends across all talks.

## 19. Extension: subset the vectors and refit

Filtering asks where selected talks lie in the original map. Refitting asks for a new view of variation within that subset. These answer different questions. Start with a sufficiently large subset, such as our 2023-and-later records, rather than only two or three talks.


In [ ]:
# TODO: Subset X using keep.to_numpy().
# Fit a new PHATE instance on those vectors.
# Store and plot the new coordinates, keeping the old coordinates too.

**Discuss:** Which relationships seem to persist? The refitted map may rotate, reflect, rescale, and change shape. Its coordinates are not directly comparable with coordinates from the full fit. Check the actual texts and their original vectors rather than interpreting movement across maps as change in the talks.

## Finish the discussion here

- What question did the next plot help you answer?
- Which kickers support your explanation, and which complicate it?
- What did filtering do? What changed when we refitted?
- What can the short kicker and selected sample leave out?

State one finding using this pattern: **In this selected sample, we noticed ___; the kickers ___ support this, while ___ complicates it. Our next check would be ___.** There is no need to return to the slides.

## Optional: save the real vectors for later

After completing the demonstration, save a single file. Keep row identifiers and texts with the vectors so a later notebook can check their alignment. This is introduced after the visible embedding loop, not hidden inside it. The practice notebook can then work without a running Ollama model.


In [ ]:
np.savez_compressed(
    "data/talk_embeddings.npz",
    embeddings=X,
    talk_id=df["talk_id"].to_numpy(dtype=str),
    kicker=df["kicker"].to_numpy(dtype=str),
    model=np.array(MODEL),
    prefix=np.array("clustering: "),
)

To reuse those vectors in a later session after loading the same CSV:

```python
saved = np.load("data/talk_embeddings.npz", allow_pickle=False)
assert saved["model"].item() == MODEL
assert saved["prefix"].item() == "clustering: "
assert np.array_equal(saved["talk_id"], df["talk_id"].to_numpy(dtype=str))
assert np.array_equal(saved["kicker"], df["kicker"].to_numpy(dtype=str))
X = saved["embeddings"]
```

A different model, changed text, or a changed row order requires a matching set of vectors.

<a id="publish-plotly"></a>

## 20. Publish a Plotly figure inside a Quarto page

Export the full-data figure constructed before filtering. Its JavaScript is included, so it can be opened independently of Python or hosted on GitHub Pages. This figure supports hover, pan, zoom, and legend interaction; filtering remains ordinary pandas code in the notebook.

In [ ]:
full_map.write_html("talk_map.html", include_plotlyjs=True)

### Embed the exported figure in your `.qmd`

1. Run the export cell above. Keep `talk_map.html` next to `index.qmd` at the project root, rather than putting the only copy inside `docs/`.
2. Paste the following into `index.qmd` (or another `.qmd` in the same folder):

````markdown
```{=html}
<iframe src="talk_map.html" width="100%" height="650"
        style="border: 0;" title="Interactive conference talk map"></iframe>
```
````

3. This project already has the following entry under `project.resources` in `_quarto.yml`:

```yaml
project:
  resources:
    - talk_map*.html
```

For a different project, merge that resource entry into its existing `project` configuration. Do not replace the rest of its configuration. Quarto copies the exported HTML into the rendered website so the relative iframe URL also works on GitHub Pages.

4. Save the files, then run from the project terminal:

```bash
quarto preview
```

No Ollama request is made by previewing the page: the figure is already exported. After editing the plot, rerun its export cell and preview again. To publish, run `quarto render`, commit the updated `docs/`, and use the repository's `/docs` GitHub Pages source.

The homepage also includes a tiny three-point publishing example that works immediately without model inference. Once your real talk map exists, use the same iframe pattern with `talk_map.html`.


## References

- [Nomic model card and task prefixes](https://huggingface.co/nomic-ai/nomic-embed-text-v1.5)
- [Ollama embedding API](https://docs.ollama.com/api/embed)
- [PHATE API](https://phate.readthedocs.io/en/stable/api.html)
- [Plotly scatterplots](https://plotly.com/python/line-and-scatter/) and [interactive HTML export](https://plotly.com/python/interactive-html-export/)
- [Data notes and source URLs](data/README.md)
- [Quarto raw HTML](https://quarto.org/docs/authoring/markdown-basics.html#raw-content) and [project resources](https://quarto.org/docs/projects/quarto-projects.html#project-resources)